[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 Hard: GPT-2 Transformer Block

Implement a full **GPT-2 style Transformer block** — combining everything you've learned.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import math

In the context of LayerNorm, \(d\) (or normalized_shape) represents the number of dimensions—specifically the feature dimension—that the model will normalize over.

In [10]:
# ✏️ YOUR IMPLEMENTATION HERE

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_model = d_model
        self.num_heads = num_heads
        self.mlp = nn.Sequential(nn.Linear(d_model, 4*d_model), nn.GELU(), nn.Linear(4*d_model, d_model))
        pass  # Initialize layers

    def forward(self, x):
        h = self.ln1(x)
        Q = self.W_q(h)
        K = self.W_k(h)
        V = self.W_v(h)
        
        seq_len = Q.shape[1]
        scores = torch.bmm(Q, torch.transpose(K, 1, 2))/math.sqrt(self.d_model)
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
        scores.masked_fill_(mask.bool(), -float('inf')) 
        attn = torch.bmm(torch.softmax(scores, dim = -1), V)
        x = x + self.W_o(attn)
        x = x + self.mlp(self.ln2(x))
        return x


        pass  # Pre-norm + causal attention + MLP with residuals

In [11]:
# 🧪 Debug
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("Output shape:", out.shape)           # (2, 8, 64)
print("Is nn.Module?", isinstance(block, nn.Module))
print("Params:", sum(p.numel() for p in block.parameters()))

Output shape: torch.Size([2, 8, 64])
Is nn.Module? True
Params: 49984


In [12]:
from torch_judge import check
check('gpt2_block')


🧪 Testing: GPT-2 Transformer Block (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (6.0ms)
  ✅ [2/5] Has LayerNorm (pre-norm architecture) (0.9ms)
  ✅ [3/5] MLP has 4x expansion with GELU (1.0ms)
  ✅ [4/5] Causal masking — future doesn't affect past (2.8ms)
  ✅ [5/5] Gradient flow to all parameters (3.3ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (14.0ms total)
  Progress saved. Run status() to see your dashboard.

